# Notebook 07: Over-fitting check on a new day, Friday afternoon (Stage 7)

**Why:** on Wednesday's own test set the trees score about 99.9% accuracy. That test
set comes from the same day as the training set (split at random), so the score may be
far too optimistic: the trees may have learned *Wednesday*, not attacks. Here the **exact
same trees** (trained on the Wednesday training set, split 42, as in notebooks 4-5) are
run on the **Friday afternoon DDoS** file of CICIDS2017, which they have never seen.

**Why Friday afternoon?** It is the best over-fitting check CICIDS2017 offers:
- It has a lot of **normal traffic** (97,718 flows before cleaning), so we can see whether
  false alarms rise on a new day.
- Its attack is a **DDoS HTTP flood** (128,027 flows before cleaning, LOIT tool) against
  port 80 of the **same web server** (192.168.10.50) that Wednesday's four DoS attacks
  targeted. It is the closest relative of Wednesday's DoS attacks on any other day, but
  it was sent from three different attacking machines with a different tool.
- No other CICIDS2017 day has Wednesday's exact attacks, so no day can test them directly.

The trees cannot output the name "DDoS" (they never saw it), so four numbers are reported:

| Measure | What it means here |
|---|---|
| **Accuracy (exact label)** | Share of flows given exactly the right label, as on Wednesday. Every DDoS flow counts as wrong, because "DDoS" is not a label the tree knows. |
| **Accuracy (attack vs normal)** | Share of flows put on the right side: normal called normal, attack called *any* attack. |
| **False-alarm rate** | Share of Friday's normal flows called an attack. Normal traffic exists on both days, so this is the cleanest over-fitting test. |
| **Attack detection rate** | Share of Friday's DDoS flows called *any* attack. |

Every depth limit from notebook 05 is tested, for all 8 feature lists, to see whether
smaller trees transfer to a new day better or worse.

**Inputs:** `Wednesday-workingHours.pcap_ISCX.csv` and
`Friday-WorkingHours-Afternoon-DDos.pcap_ISCX.csv` in `data/` (local) or the Drive
`data/` folder (Colab). **Runtime:** about 10 minutes on a 2-core CPU.

In [1]:
import json
import time
import datetime
from pathlib import Path

import numpy as np
import pandas as pd
import sklearn
from sklearn.model_selection import train_test_split
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import accuracy_score

try:
    from google.colab import drive
    drive.mount("/content/drive")
    PROJECT_DIR = Path("/content/drive/MyDrive/Colab Notebooks/threat detector")
    REPO_DIR = PROJECT_DIR
except ImportError:
    REPO_DIR = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
    PROJECT_DIR = REPO_DIR
DATA_DIR = PROJECT_DIR / "data"
OUT_DIR = REPO_DIR / "experiments"
FEATURE_SETS_JSON = REPO_DIR / "experiments" / "feature_sets.json"
if not FEATURE_SETS_JSON.exists():
    FEATURE_SETS_JSON = DATA_DIR / "feature_sets.json"

FILES = {
    "Wednesday test set": "Wednesday-workingHours.pcap_ISCX.csv",
    "Friday afternoon": "Friday-WorkingHours-Afternoon-DDos.pcap_ISCX.csv",
}
DEPTHS = [1, 2, 3, 4, 5, 6, 8, 10, 12, 15, 20, None]
SEED = 42
BENIGN = "BENIGN"
print("scikit-learn", sklearn.__version__, "| data folder:", DATA_DIR)

scikit-learn 1.9.1 | data folder: /home/user/threat-detection-model/data


## Load and clean each file exactly like notebook 2
(strip column names, drop rows with infinite values, drop rows with missing values,
drop exact duplicates).

In [2]:
def load_clean(filename):
    df = pd.read_csv(DATA_DIR / filename)
    df.columns = df.columns.str.strip()
    n0 = len(df)
    num = df.select_dtypes(include=np.number).columns
    df = df[~np.isinf(df[num]).any(axis=1)]
    n_inf = n0 - len(df)
    n1 = len(df)
    df = df.dropna()
    n_na = n1 - len(df)
    n2 = len(df)
    df = df.drop_duplicates()
    n_dup = n2 - len(df)
    df["Label"] = df["Label"].str.strip()
    print(f"{filename}: {n0:,} rows -> {len(df):,} after removing {n_inf:,} infinite, "
          f"{n_na:,} missing, {n_dup:,} duplicate rows")
    return df


wed = load_clean(FILES["Wednesday test set"])
X = wed.drop(columns=["Label"])
y = wed["Label"]
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=SEED, stratify=y)
assert X_train.shape == (488393, 78) and X_test.shape == (122099, 78)   # same split as notebooks 2-5

tests = {"Wednesday test set": (X_test, y_test)}
for name in ["Friday afternoon"]:
    d = load_clean(FILES[name])
    assert list(d.drop(columns=["Label"]).columns) == list(X.columns), "columns differ from Wednesday"
    tests[name] = (d.drop(columns=["Label"]), d["Label"])
for name, (_, yy) in tests.items():
    print(f"\n{name}:\n{yy.value_counts().to_string()}")

Wednesday-workingHours.pcap_ISCX.csv: 692,703 rows -> 610,492 after removing 1,297 infinite, 0 missing, 80,914 duplicate rows


Friday-WorkingHours-Afternoon-DDos.pcap_ISCX.csv: 225,745 rows -> 223,082 after removing 34 infinite, 0 missing, 2,629 duplicate rows

Wednesday test set:
Label
BENIGN              83348
DoS Hulk            34569
DoS GoldenEye        2057
DoS slowloris        1077
DoS Slowhttptest     1046
Heartbleed              2

Friday afternoon:
Label
DDoS      128014
BENIGN     95068


## Feature lists (same as notebook 05) and the depths chosen on validation there

In [3]:
with open(FEATURE_SETS_JSON) as f:
    base = json.load(f)
SHORTCUT = ["Destination Port", "Init_Win_bytes_forward", "Init_Win_bytes_backward"]
FEATURE_SETS = {k: base[k] for k in "ABCD"}
FEATURE_SETS.update({f"{k}-NS": [c for c in base[k] if c not in SHORTCUT] for k in "ABCD"})
choice_path = OUT_DIR / "S6_validation_choice.csv"
CHOSEN = (pd.read_csv(choice_path, dtype={"chosen_depth_1pt": str}).set_index("feature_set")["chosen_depth_1pt"].to_dict()
          if choice_path.exists() else {})
print({k: len(v) for k, v in FEATURE_SETS.items()})
print("Depths chosen on validation in notebook 05:", CHOSEN)

{'A': 58, 'B': 39, 'C': 15, 'D': 6, 'A-NS': 55, 'B-NS': 36, 'C-NS': 13, 'D-NS': 4}
Depths chosen on validation in notebook 05: {'A': '20', 'B': '20', 'C': '12', 'D': '12', 'A-NS': '20', 'B-NS': '15', 'C-NS': '12', 'D-NS': '20'}


## Train on the Wednesday training set, test on both files

In [4]:
def score(y_true, y_pred):
    y_true = np.asarray(y_true)
    real_attack = y_true != BENIGN
    said_attack = y_pred != BENIGN
    out = {
        "rows": len(y_true),
        "benign_rows": int((~real_attack).sum()),
        "attack_rows": int(real_attack.sum()),
        "accuracy_exact_label": accuracy_score(y_true, y_pred),
        "accuracy_attack_vs_normal": float((real_attack == said_attack).mean()),
        "false_alarms": int((~real_attack & said_attack).sum()),
        "false_alarm_rate": float((~real_attack & said_attack).sum() / max((~real_attack).sum(), 1)),
        "attacks_flagged": int((real_attack & said_attack).sum()),
        "attack_detection_rate": float((real_attack & said_attack).sum() / real_attack.sum()) if real_attack.any() else np.nan,
    }
    # what the false alarms were called
    fa = pd.Series(y_pred[~real_attack & said_attack]).value_counts()
    out["false_alarms_called"] = "; ".join(f"{k} {v}" for k, v in fa.items())
    # how many of each attack type were flagged as any attack
    for lab in sorted(set(y_true[real_attack])):
        m = y_true == lab
        out[f"flagged_{lab.replace(' ', '_')}"] = f"{int(said_attack[m].sum())}/{int(m.sum())}"
    return out


rows = []
t0 = time.perf_counter()
for set_name, cols in FEATURE_SETS.items():
    for d in DEPTHS:
        model = DecisionTreeClassifier(random_state=SEED, max_depth=d).fit(X_train[cols], y_train)
        for test_name, (Xt, yt) in tests.items():
            y_pred = model.predict(Xt[cols])
            rows.append({"feature_set": set_name, "num_features": len(cols),
                         "max_depth": "none" if d is None else str(d), "nodes": int(model.tree_.node_count),
                         "depth_reached": int(model.get_depth()),
                         "test_file": test_name, **score(yt, y_pred)})
    print(f"{set_name} done ({time.perf_counter() - t0:.0f} s)")
res = pd.DataFrame(rows)
res.to_csv(OUT_DIR / "S7_friday_test.csv", index=False)
print("Saved", OUT_DIR / "S7_friday_test.csv")

A done (134 s)


B done (225 s)


C done (270 s)


D done (292 s)


A-NS done (421 s)


B-NS done (510 s)


C-NS done (554 s)


D-NS done (580 s)
Saved /home/user/threat-detection-model/experiments/S7_friday_test.csv


## Summary: unlimited trees and the depths chosen on validation

In [5]:
pick = res[(res["max_depth"] == "none")
           | res.apply(lambda r: CHOSEN.get(r["feature_set"]) == r["max_depth"], axis=1)].copy()
pick["tree"] = np.where(pick["max_depth"] == "none", "unlimited", "chosen depth " + pick["max_depth"])
cols_show = ["feature_set", "tree", "test_file", "accuracy_exact_label", "accuracy_attack_vs_normal",
             "false_alarm_rate", "false_alarms", "attack_detection_rate", "attacks_flagged", "attack_rows"]
summary = pick[cols_show].sort_values(["feature_set", "tree", "test_file"])
summary.to_csv(OUT_DIR / "S7_friday_summary.csv", index=False)
pd.set_option("display.width", 250)
display(summary.round(5))

,feature_set,tree,test_file,accuracy_exact_label,accuracy_attack_vs_normal,false_alarm_rate,false_alarms,attack_detection_rate,attacks_flagged,attack_rows
21,A,chosen depth 20,Friday afternoon,0.42605,0.79010,0.00026,25,0.63441,81214,128014
20,A,chosen depth 20,Wednesday test set,0.99948,0.99959,0.00025,21,0.99925,38722,38751
23,A,unlimited,Friday afternoon,0.42605,0.59666,0.00025,24,0.29730,38059,128014
22,A,unlimited,Wednesday test set,0.99943,0.99953,0.00025,21,0.99907,38715,38751
117,A-NS,chosen depth 20,Friday afternoon,0.42542,0.60526,0.00173,164,0.31340,40119,128014
116,A-NS,chosen depth 20,Wednesday test set,0.99722,0.99735,0.00122,102,0.99427,38529,38751
119,A-NS,unlimited,Friday afternoon,0.42534,0.64886,0.00191,182,0.38951,49863,128014
118,A-NS,unlimited,Wednesday test set,0.99698,0.99712,0.00134,112,0.99381,38511,38751
45,B,chosen depth 20,Friday afternoon,0.42590,0.59682,0.00061,58,0.29785,38129,128014
44,B,chosen depth 20,Wednesday test set,0.99941,0.99952,0.00031,26,0.99917,38719,38751


## Does a smaller tree transfer better to a new day?
False-alarm rate on Friday's normal traffic, and the share of Friday's DDoS flows flagged,
at every depth (percent).

In [6]:
order = ["none" if d is None else str(d) for d in DEPTHS]
fri = res[res["test_file"] == "Friday afternoon"].set_index(["feature_set", "max_depth"])
print("False-alarm rate on Friday's normal traffic (%):")
display((100 * fri["false_alarm_rate"]).unstack(0).loc[order].round(3))
print("Share of Friday's DDoS flows flagged as an attack (%):")
display((100 * fri["attack_detection_rate"]).unstack(0).loc[order].round(2))

False-alarm rate on Friday's normal traffic (%):


feature_set,A,A-NS,B,B-NS,C,C-NS,D,D-NS
max_depth,,,,,,,,
1,0.088,0.088,0.088,0.088,15.188,15.188,9.003,9.003
2,0.088,0.088,0.088,0.088,0.045,0.045,7.734,7.734
3,3.096,0.167,4.929,2.001,3.075,0.308,0.620,7.906
4,0.992,0.183,0.990,0.014,1.097,0.380,0.484,7.908
5,1.181,0.298,1.179,0.116,1.173,0.434,2.485,9.598
6,1.047,0.162,1.047,0.540,1.228,0.352,2.836,9.064
8,0.491,3.522,0.496,0.121,0.566,0.199,0.522,9.634
10,0.043,0.368,0.054,0.405,0.082,0.177,0.351,10.000
12,0.021,0.119,0.066,0.159,0.082,0.459,0.300,9.740


Share of Friday's DDoS flows flagged as an attack (%):


feature_set,A,A-NS,B,B-NS,C,C-NS,D,D-NS
max_depth,,,,,,,,
1,62.69,62.69,62.69,62.69,63.39,63.39,16.97,16.97
2,62.69,62.69,62.69,62.69,63.39,63.39,16.67,16.67
3,62.73,62.69,62.77,62.73,63.43,63.39,16.97,16.71
4,29.20,29.20,38.41,29.20,63.39,63.39,17.09,16.71
5,30.85,29.20,29.23,29.21,63.40,63.40,17.33,17.66
6,62.72,30.83,29.35,38.54,63.44,63.42,17.40,18.33
8,31.31,31.29,63.17,29.66,18.13,16.66,17.18,33.08
10,31.30,29.65,29.69,38.88,18.14,60.62,17.42,52.18
12,31.33,38.88,29.69,38.82,16.83,61.66,17.11,53.34


In [7]:
# Rows for results.csv (one per feature list: the unlimited tree on the Friday file)
today = datetime.date.today().isoformat()
out_rows = []
for set_name in FEATURE_SETS:
    r = res[(res["feature_set"] == set_name) & (res["max_depth"] == "none") & (res["test_file"] == "Friday afternoon")]
    n = r["rows"].sum()
    acc = (r["accuracy_exact_label"] * r["rows"]).sum() / n
    acc_bin = (r["accuracy_attack_vs_normal"] * r["rows"]).sum() / n
    fa, ben = r["false_alarms"].sum(), r["benign_rows"].sum()
    caught, att = r["attacks_flagged"].sum(), r["attack_rows"].sum()
    out_rows.append({
        "experiment_id": f"S7-{set_name}-friday", "date": today, "stage": "7-new-day-test",
        "dataset_file": "trained: Wednesday (split 42 training set); tested: Friday-WorkingHours-Afternoon-DDos",
        "num_samples": int(n), "num_features": len(FEATURE_SETS[set_name]), "features_used": set_name,
        "max_depth": int(r["depth_reached"].iloc[0]), "num_nodes": int(r["nodes"].iloc[0]),
        "accuracy": round(acc, 6), "precision": "", "recall": "", "f1": "", "model_size_kb": "",
        "training_time_s": "", "inference_ms": "", "ram_mb": "",
        "notes": (f"unlimited tree trained on Wednesday, tested on the Friday afternoon DDoS file ({n} cleaned flows); "
                  f"accuracy = exact label (every DDoS flow counts as wrong, since the label DDoS was never "
                  f"seen in training); accuracy attack-vs-normal={acc_bin:.6f}; false alarms={fa} of {ben} "
                  f"benign flows ({100 * fa / ben:.3f}%); DDoS flows flagged as an attack={caught} of {att}"),
    })
s7 = pd.DataFrame(out_rows)
results = pd.read_csv(OUT_DIR / "results.csv", dtype=str)
results = results[~results["experiment_id"].str.startswith("S7-")]
pd.concat([results, s7.astype(str)]).to_csv(OUT_DIR / "results.csv", index=False)
display(s7.drop(columns=["notes"]))

,experiment_id,date,stage,dataset_file,num_samples,num_features,features_used,max_depth,num_nodes,accuracy,precision,recall,f1,model_size_kb,training_time_s,inference_ms,ram_mb
0,S7-A-friday,2026-10-29,7-new-day-test,trained: Wednesday (split 42 training set); te...,223082,58,A,31,823,0.426050,,,,,,,
1,S7-B-friday,2026-10-29,7-new-day-test,trained: Wednesday (split 42 training set); te...,223082,39,B,31,891,0.425982,,,,,,,
2,S7-C-friday,2026-10-29,7-new-day-test,trained: Wednesday (split 42 training set); te...,223082,15,C,31,997,0.425799,,,,,,,
3,S7-D-friday,2026-10-29,7-new-day-test,trained: Wednesday (split 42 training set); te...,223082,6,D,32,1487,0.425583,,,,,,,
4,S7-A-NS-friday,2026-10-29,7-new-day-test,trained: Wednesday (split 42 training set); te...,223082,55,A-NS,51,2859,0.425341,,,,,,,
5,S7-B-NS-friday,2026-10-29,7-new-day-test,trained: Wednesday (split 42 training set); te...,223082,36,B-NS,51,3439,0.425099,,,,,,,
6,S7-C-NS-friday,2026-10-29,7-new-day-test,trained: Wednesday (split 42 training set); te...,223082,13,C-NS,43,4003,0.424620,,,,,,,
7,S7-D-NS-friday,2026-10-29,7-new-day-test,trained: Wednesday (split 42 training set); te...,223082,4,D-NS,67,16233,0.389870,,,,,,,
